# S02 · Structured Outputs and Model Selection

## Why this notebook exists

Northstar wants software that reads a support ticket and returns four things: a category, a
priority, whether a person should review it, and a short summary. The answer must always use the
same fields so other software can read it safely. It must also make sensible decisions—for example,
a critical outage should not be marked as low priority.

We will compare two pretend model options. One is faster and cheaper but sometimes misses urgent
tickets. The other is slower and more careful but sometimes asks for human review when it is not
needed. Your job is to decide what Northstar should measure before choosing between them.

This notebook separates two questions:

1. **Is the answer in the required format?** This is structured-output validation.
2. **Is the answer actually correct for the ticket?** This is model evaluation.

An answer can pass the first check and still fail the second. `{"priority": "low"}` may be valid
JSON with an allowed value, but it is still wrong for a critical outage.

## What you will learn

1. Explain the different jobs of the **model**, **API**, and **application**.
2. Use Pydantic to describe and check the required answer format.
3. Explain why a correctly formatted answer can still be wrong.
4. Test two model options and inspect the kinds of mistakes each one makes.
5. Recommend an option using correctness, speed, cost, and the seriousness of its mistakes.


## How the lab fits together

First, you will see which jobs belong to the model, the API, and Northstar's own application. Next,
you will define the required answer format and try examples that pass and fail. You will then test
two simulated model options, inspect their mistakes, estimate cost, and write a recommendation.
The examples are fictional and repeatable; they do not describe any real model's performance.

> [!IMPORTANT]
> Each ticket is handled with one model request. This notebook does not use tools or build an agent
> loop; S01 covers that separate pattern.


## 1. 🧩 Three layers: do not mix responsibilities

**Goal:** identify which system component owns each responsibility. This distinction helps you
diagnose failures accurately instead of treating every problem as a 'model problem.'

```text
Northstar UI / business workflow  →  model-serving API  →  model
  inputs, policy, validation          request/response      generated content
```

- The **model** generates content.
- The **serving API** transports requests and may offer features such as schema-constrained output.
- The **application** supplies context, validates results, applies policy, handles errors, and owns
  the customer impact.

A provider may supply both the model and API, but that does not transfer the application's
responsibility for authorization or business decisions.

**Prediction:** A response conforms to JSON Schema but assigns the wrong urgency. Is that an API contract problem, an outcome-quality problem, or both?

## 2. 💻 Define a contract with Pydantic

**Goal:** define exactly what downstream code is allowed to receive. The `TicketDecision` model
specifies required fields, permitted labels, and basic value constraints. Pydantic then converts
valid input into a typed object or raises a validation error.

A typed output is a **data contract**, not a guarantee of correct classification. It can reject
`priority="urgent"` when only low/medium/high are allowed, but it cannot know whether a valid
`priority="low"` is appropriate for the ticket.

> [!TIP]
> **📌 Reminder:** a typed contract is a validation rule, not a correctness guarantee —
> keep that distinction in mind for every cell below.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError, ConfigDict

class TicketDecision(BaseModel):
    model_config = ConfigDict(extra="forbid")
    category: Literal["billing", "technical", "access", "general"]
    priority: Literal["low", "medium", "high"]
    needs_human: bool
    summary: str = Field(min_length=1, max_length=240)

valid = TicketDecision.model_validate({
    "category": "technical", "priority": "high", "needs_human": True,
    "summary": "Repeated sign-in failures block the customer."
})
print(valid.model_dump())

**Predict before running:** What happens for `priority="urgent"`? What if `needs_human="false"` is passed as a string? Pydantic may coerce some types; `Literal` rejects unknown priority labels. A stricter application can use strict types or extra validators.

In [ ]:
bad = {"category": "technical", "priority": "urgent", "needs_human": False,
       "summary": "User cannot sign in."}
try:
    TicketDecision.model_validate(bad)
except ValidationError as exc:
    print("Rejected field(s):", sorted({str(err["loc"][0]) for err in exc.errors()}))

print("Contract fields:", list(TicketDecision.model_json_schema()["properties"]))

### 🛠️ Mini exercise
Create a case that **passes** schema validation but is **wrong** for the ticket described. Explain why structured output alone cannot check business truth. Optionally add a `ticket_id` field and try validating again.

## 3. 🧩 Synthetic Northstar support data

**Goal:** understand that an evaluation is only as credible as its examples and expected labels.
The tickets below include both clear and ambiguous cases so you can inspect different failure
types. The labels are **course examples**, not human-reviewed production ground truth.

Before evaluating a candidate, read several rows and ask whether the expected category, priority,
and escalation decision are supported by the ticket text. Disagreement here is a dataset issue,
not evidence that one candidate is wrong.

In [ ]:
tickets = [
    {"id":"N-001", "text":"Invoice charged twice for last month.", "expected_category":"billing", "expected_priority":"medium", "expected_human":True},
    {"id":"N-002", "text":"Critical outage: all administrators get 503 errors.", "expected_category":"technical", "expected_priority":"high", "expected_human":True},
    {"id":"N-003", "text":"Where can I find the API documentation?", "expected_category":"general", "expected_priority":"low", "expected_human":False},
    {"id":"N-004", "text":"Unable to log in after password reset.", "expected_category":"technical", "expected_priority":"medium", "expected_human":False},
    {"id":"N-005", "text":"Please update the company name on our receipt.", "expected_category":"billing", "expected_priority":"low", "expected_human":False},
    {"id":"N-006", "text":"Unexpected $900 charge; finance closes books today.", "expected_category":"billing", "expected_priority":"high", "expected_human":True},
    {"id":"N-007", "text":"How do I invite a teammate?", "expected_category":"general", "expected_priority":"low", "expected_human":False},
    {"id":"N-008", "text":"API intermittently times out for our integration.", "expected_category":"technical", "expected_priority":"medium", "expected_human":False},
    {"id":"N-009", "text":"Need a refund for a canceled subscription.", "expected_category":"billing", "expected_priority":"medium", "expected_human":True},
    {"id":"N-010", "text":"The dashboard charts look blurry on one browser.", "expected_category":"technical", "expected_priority":"low", "expected_human":False},
    {"id":"N-011", "text":"Security incident reported; need an engineer now.", "expected_category":"technical", "expected_priority":"high", "expected_human":True},
    {"id":"N-012", "text":"Need the contract contact; I can't find an owner.", "expected_category":"general", "expected_priority":"medium", "expected_human":True},
]
print(f"Synthetic tickets: {len(tickets)}")
for t in tickets[:3]:
    print(t["id"], t["text"])

**Discuss:** What else would Northstar need to establish a credible evaluation set? Consider account tier, policy version, label-review process, distribution shifts, and the cost of a false low-priority decision.

## 4. 🔍 Compare two *simulated* configurations (offline)

**Goal:** create controlled candidate behavior so you can learn the evaluation method. Each
configuration makes repeatable mistakes, allowing you to connect a metric back to specific rows.

The following deterministic functions are **teaching stand-ins**—one behaves like a fast, cheap
candidate and the other like a cautious candidate. They are **not real LLMs** and do not estimate
any provider's accuracy, latency, or price.

In [ ]:
def make_decision(ticket, config):
    # An offline simulation: an intentionally imperfect output table.
    by_id = {t["id"]: t for t in tickets}
    target = by_id[ticket["id"]]
    pred = {
        "category": target["expected_category"],
        "priority": target["expected_priority"],
        "needs_human": target["expected_human"],
        "summary": "Synthetic baseline decision for teaching evaluation.",
    }
    if config == "fast_simulated":
        if ticket["id"] in {"N-002", "N-006", "N-011"}:
            pred["priority"] = "medium"  # risky false downgrades
        if ticket["id"] in {"N-005", "N-012"}:
            pred["category"] = "general"
    elif config == "careful_simulated":
        if ticket["id"] == "N-003":
            pred["needs_human"] = True  # unnecessary escalation
        if ticket["id"] == "N-008":
            pred["priority"] = "high"  # over-prioritization
    else:
        raise ValueError(f"Unknown simulated config: {config}")
    return TicketDecision.model_validate(pred)

for config in ("fast_simulated", "careful_simulated"):
    print(config, make_decision(tickets[0], config).model_dump(exclude={"summary"}))

## 5. 🔍 Evaluate business correctness, not merely format

**Goal:** compare candidates using both an aggregate score and the consequences of individual
errors. Exact-match accuracy requires category, priority, and escalation to all match. We also
count high-priority downgrades because Northstar considers those more harmful than unnecessary
human review.

When you inspect the results, look past the totals: Which tickets failed? What type of error was
made? Who bears the cost? Twelve synthetic examples are enough to demonstrate the method, but not
enough to establish real-world performance.

In [ ]:
PRIORITY_RANK = {"low": 0, "medium": 1, "high": 2}

def evaluate(config):
    rows = []
    for ticket in tickets:
        pred = make_decision(ticket, config)
        exact = (pred.category == ticket["expected_category"]
                 and pred.priority == ticket["expected_priority"]
                 and pred.needs_human == ticket["expected_human"])
        high_downgrade = (ticket["expected_priority"] == "high"
                          and PRIORITY_RANK[pred.priority] < PRIORITY_RANK["high"])
        rows.append({"id": ticket["id"], "exact": exact,
                     "high_priority_downgrade": high_downgrade,
                     "human_mismatch": pred.needs_human != ticket["expected_human"]})
    return {
        "config": config,
        "exact_matches": sum(r["exact"] for r in rows),
        "total": len(rows),
        "high_priority_downgrades": sum(r["high_priority_downgrade"] for r in rows),
        "human_flag_mismatches": sum(r["human_mismatch"] for r in rows),
        "incorrect_ids": [r["id"] for r in rows if not r["exact"]],
    }

results = [evaluate(c) for c in ("fast_simulated", "careful_simulated")]
for result in results:
    print(result)

**Interpretation prompt:** Is the candidate with fewer total mistakes always the better choice? Compare serious *under-prioritization* with unnecessary human escalation.

### 🛠️ Learner modification
1. Add one ambiguous ticket and a proposed expected result.
2. Introduce one new simulated failure case in `make_decision`.
3. Re-run evaluation; explain whether aggregate accuracy hides a high-impact error.

**Debugging task:** Remove the mapping for a configuration or change a priority to `"urgent"`; observe whether you get a runtime error or validation error, then fix it.

## 6. 🧩 Tokens, latency, and cost

**Goal:** understand why the cheapest price per token is not the same as the cheapest reliable
application. A configuration that retries often, produces long answers, or sends more cases to
human review may cost more overall even when its advertised token rate is lower.

Actual cost varies by vendor, model, output length, cache behavior, region, and date. The formula
below illustrates billing mechanics, not current vendor prices.

Cost estimate = `(input_tokens / 1_000_000 * input_rate) + (output_tokens / 1_000_000 * output_rate)`.

Reasoning tokens, cached-token rates, tool usage, minimum fees, and retries may change actual billed amounts. Use provider usage fields rather than assuming counts from English word totals.

In [ ]:
def estimate_usd(input_tokens, output_tokens, input_usd_per_m, output_usd_per_m):
    return (input_tokens * input_usd_per_m + output_tokens * output_usd_per_m) / 1_000_000

example_cost = estimate_usd(1200, 180, 1.00, 4.00)  # hypothetical rates only
print(f"Illustrative single-call cost: ${example_cost:.6f}")
print(f"Illustrative 10,000-call cost: ${example_cost * 10_000:.2f}")

## 7. Optional live-model comparison

**Goal:** replace a simulated candidate with one measured provider response while keeping the same validation boundary. The returned JSON is parsed with Pydantic just like the offline examples. Requesting JSON format helps with syntax, but it is **not the same as enforcing the full Pydantic contract during generation**, and neither mechanism guarantees a correct classification.

> [!CAUTION]
> This optional section can incur provider charges and requires `OPENROUTER_API_KEY` plus a supported model ID. Use only synthetic tickets, review the provider's privacy and pricing terms, and keep `RUN_LIVE=False` unless you intentionally want to make the request. Provider capabilities change, so verify structured-output support for the selected route.

In [ ]:
RUN_LIVE = False  # explicit opt-in
LIVE_MODEL = "REPLACE_WITH_SUPPORTED_MODEL_ID"

if RUN_LIVE:
    import json
    import os
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv()  # load credentials without placing them in notebook cells
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("Set OPENROUTER_API_KEY outside the notebook")
    if LIVE_MODEL.startswith("REPLACE_"):
        raise RuntimeError("Set LIVE_MODEL to a verified OpenRouter model ID")

    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
    ticket = tickets[0]  # synthetic sample, one call only
    response = client.chat.completions.create(
        model=LIVE_MODEL,
        messages=[
            {"role":"system", "content":(
                "You triage synthetic support tickets. Reply with a JSON object containing "
                "category (billing/technical/access/general), priority (low/medium/high), "
                "needs_human (boolean), and summary (brief string).")},
            {"role":"user", "content":ticket["text"]}
        ],
        response_format={"type":"json_object"},
    )
    raw = response.choices[0].message.content
    live_decision = TicketDecision.model_validate(json.loads(raw))
    print(live_decision.model_dump())
    print("Provider reported usage:", response.usage)
else:
    print("Live API call skipped (RUN_LIVE=False).")


### Optional model comparison worksheet
After verifying compatibility, compare **the same tickets** with two configurations. Record actual measurements instead of copying the offline simulation as if it were model performance.

| Candidate | Exact match | High-priority downgrades | Median/p95 latency | Input/output tokens | Total cost | Failures / retries |
|---|---:|---:|---:|---:|---:|---|
| Configuration A | — | — | — | — | — | — |
| Configuration B | — | — | — | — | — | — |

For credible decisions: establish a reviewed test set; repeat trials where output varies; consider confidence intervals and slice-level error rates; avoid choosing based on a single small run. For production decisions, consider security, privacy, and data residency.

## 8. 🛠️ Make the deployment recommendation

**Goal:** synthesize contract validity, outcome quality, risk, latency, and cost into a defensible
recommendation. There may not be one universally best model; the right choice depends on the
customer's minimum quality threshold and the cost of each failure type.
**Customer context:** Northstar says urgent incidents cannot be under-prioritized, but human review time is limited. A stakeholder asks you to choose the cheapest model.

Respond in 3–5 sentences:
- What is your **minimum quality criterion** (including high-severity error policy)?
- Which measurements would you collect on a representative dataset?
- What do you communicate about the reliability of the 12-ticket synthetic comparison?
- What is your fallback when schema parsing fails or information is insufficient?


## 9. ✅ Check your understanding
1. Why can a perfectly valid JSON result still be a bad answer?
2. Which layer should own escalation rules and audit logging?
3. What are the risks of testing on the examples used to design the prompt?
4. Does increasing reasoning effort necessarily improve classification accuracy?
5. Which is worse for this customer: a false high-priority downgrade or a false human escalation, and why?

## 10. ✅ Recap

**You've now done what Session 02 promises:** built a typed contract, caught a schema
violation on purpose, and compared two configurations by *failure type* instead of raw
accuracy. Carry these into every future model-selection decision:

- State the distinction between **contract compliance**, **correctness**, and **business utility**.
- Talk about **error slices** and asymmetric costs, not aggregate accuracy alone.
- Explain how token usage, caching, concurrency, and retries affect observed costs.
- Admit uncertainty when labels or model-usage data are insufficient.

> [!NOTE]
> **Related session:** S01 (Verified Agent Loops) covers turning tool requests and
> execution loops into an agent harness. Nothing in this notebook requires that mechanism.

### 📚 Further reading
- [Pydantic documentation](https://docs.pydantic.dev/latest/)
- [OpenRouter API reference](https://openrouter.ai/docs/api/reference/overview)
- **S01: Verified Agent Loops** — for the tool-execution loop this session deliberately does not cover.